# ChatSolveAI: Automated Customer Support with the OpenAI API
Embeddings, cosine-similarity search, and a GPT-4o-mini fallback chatbot.

In [ ]:

import os
os.environ["OPENAI_API_KEY"] = 

SyntaxError: invalid syntax (Temp/ipykernel_33988/117753976.py, line 5)

## Task 1: Knowledge base to embeddings
Output: `knowledge_embeddings.json`

In [ ]:
import json
import pandas as pd

df = pd.read_csv("knowledge_base.csv")

# One batched call for all documents, text unchanged
response = client.embeddings.create(
    model="text-embedding-3-small",
    input=df["document_text"].tolist()
)

records = []
for row, item in zip(df.itertuples(), response.data):
    records.append({
        "document_id": int(row.document_id),
        "document_text": row.document_text,
        "embedding_vector": item.embedding,
        "metadata": row.metadata
    })

with open("knowledge_embeddings.json", "w") as f:
    json.dump(records, f)

# Verify
with open("knowledge_embeddings.json") as f:
    check = json.load(f)
print(len(check), len(df))
print(check[0].keys())
print(len(check[0]["embedding_vector"]))

d:\Anaconda\lib\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (


FileNotFoundError: [Errno 2] No such file or directory: 'knowledge_base.csv'

## Task 2: Query matching with cosine similarity
Output: `query_responses.json` (top 3 responses + confidence scores, with retry on rate limits)

In [ ]:
import json, time
import numpy as np
import pandas as pd
from openai import RateLimitError, APIConnectionError, APITimeoutError

queries = pd.read_csv("processed_queries.csv")
with open("predefined_responses.json") as f:
    raw_responses = json.load(f)   # dict: category -> response text

response_texts = list(raw_responses.values())

def embed_with_retry(texts, max_retries=5, batch_size=500):
    vectors = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        for attempt in range(max_retries):
            try:
                res = client.embeddings.create(model="text-embedding-3-small", input=batch)
                vectors.extend(d.embedding for d in res.data)
                break
            except (RateLimitError, APIConnectionError, APITimeoutError):
                if attempt == max_retries - 1:
                    raise
                time.sleep(2 ** attempt)   # exponential backoff: 1, 2, 4, 8 s
    return np.array(vectors)

query_vecs = embed_with_retry(queries["query_text"].tolist())
resp_vecs = embed_with_retry(response_texts)

# Cosine similarity = dot product of normalized vectors
query_vecs /= np.linalg.norm(query_vecs, axis=1, keepdims=True)
resp_vecs /= np.linalg.norm(resp_vecs, axis=1, keepdims=True)
sims = query_vecs @ resp_vecs.T

records = []
for i, row in enumerate(queries.itertuples()):
    top_idx = np.argsort(-sims[i])[:3]
    records.append({
        "query_id": int(row.query_id),
        "query_text": row.query_text,
        "top_responses": [response_texts[j] for j in top_idx],
        "confidence_scores": [float(np.clip(sims[i][j], 0, 1)) for j in top_idx],
    })

with open("query_responses.json", "w") as f:
    json.dump(records, f, indent=4)

# Verify
with open("query_responses.json") as f:
    check = json.load(f)
print(len(check), len(queries))
print(check[0].keys())
print(check[0]["top_responses"], check[0]["confidence_scores"])

## Task 3: Chatbot with GPT fallback
Output: `sample_chatbot_responses.json`

In [ ]:
import json
import numpy as np
from datetime import datetime, timezone

with open("chatbot_responses.json") as f:
    kb = json.load(f)   # list of dicts

kb_questions = [item["query_text"] for item in kb]
kb_answers = [item["retrieved_response"] for item in kb]

def embed(texts):
    res = client.embeddings.create(model="text-embedding-3-small", input=texts)
    vecs = np.array([d.embedding for d in res.data])
    return vecs / np.linalg.norm(vecs, axis=1, keepdims=True)

kb_vecs = embed(kb_questions)   # one call for all stored questions

THRESHOLD = 0.6
conversation_history = []

def chatbot(query):
    q_vec = embed([query])[0]
    sims = kb_vecs @ q_vec
    best = int(np.argmax(sims))
    score = float(sims[best])

    if score >= THRESHOLD:
        answer = kb_answers[best]            # matched a known question
    else:
        completion = client.chat.completions.create(   # fallback: generate
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "You are a helpful customer support assistant for ChatSolveAI. Answer briefly and politely."},
                {"role": "user", "content": query},
            ],
        )
        answer = completion.choices[0].message.content

    conversation_history.append({
        "query_text": query,
        "retrieved_response": answer,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "confidence_score": round(score, 4),
    })
    return score

test_queries = [
    "When can I talk to someone from support?",          # paraphrase -> should match
    "What is your refund policy for digital products?",  # not covered -> GPT fallback
]
for q in test_queries:
    print(q, "->", chatbot(q))

with open("sample_chatbot_responses.json", "w") as f:
    json.dump(conversation_history, f, indent=4)